# Error analysis — bbox-tube-temporal

Inspect FN and FP sequences from `evaluate_packaged` output. For each
error case, renders:

1. **Raw frames** with every kept tube's bounding box overlaid,
   color-coded per tube_id.
2. **Tube timeline + filmstrips** (via `plot_tube_summary`). Filmstrip
   border is **green** if the tube crossed the classifier threshold
   (fired) and **red** otherwise.
3. **Per-tube logit legend** with the winner explicitly marked.

Requires `predictions.json` produced after the `kept_tubes` detail was
added to the deployment pipeline (i.e. a post-B1 `evaluate_packaged`
run). Configure `VARIANT`, `SPLIT`, and `PACKAGED_SUBDIR` in the next
cell to switch between baseline and Track C snapshots.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from bbox_tube_temporal.aggregation_analysis import load_predictions
from bbox_tube_temporal.tube_viz import draw_tubes_on_frames, plot_tube_summary
from bbox_tube_temporal.types import Detection, Tube, TubeEntry

VARIANT = "gru_convnext_finetune"
SPLIT = "val"
PACKAGED_SUBDIR = "packaged"
MAX_ERRORS_TO_SHOW = 10
FRAMES_PER_ROW = 5

# Paths are relative to notebooks/ (the notebook's working directory).
PREDICTIONS_PATH = Path(
    f"../data/08_reporting/{SPLIT}/{PACKAGED_SUBDIR}/{VARIANT}/predictions.json"
)
SEQUENCES_ROOT = Path(f"../data/01_raw/datasets/{SPLIT}")

In [ ]:
records = load_predictions(PREDICTIONS_PATH)
assert records and "kept_tubes" in records[0], (
    "predictions.json is missing 'kept_tubes'; re-run `dvc repro evaluate_packaged` "
    "with the post-B1 model/evaluate_packaged so per-tube details are persisted."
)

false_negatives = [r for r in records if r["label"] == "smoke" and not r["is_positive"]]
false_positives = [r for r in records if r["label"] == "fp" and r["is_positive"]]
print(f"FN: {len(false_negatives)}   FP: {len(false_positives)}")

In [ ]:
def _sequence_image_paths(label: str, sequence_id: str) -> list[Path]:
    label_dir = "wildfire" if label == "smoke" else "fp"
    seq_dir = SEQUENCES_ROOT / label_dir / sequence_id / "images"
    return sorted(seq_dir.glob("*.jpg"))


def _pad_paths_symmetrically(paths: list[Path], min_length: int) -> list[Path]:
    """Mirror symmetric padding so image indices line up with padded tubes."""
    if not paths or len(paths) >= min_length:
        return list(paths)
    result = list(paths)
    prepend = True
    while len(result) < min_length:
        if prepend:
            result.insert(0, paths[0])
        else:
            result.append(paths[-1])
        prepend = not prepend
    return result


def _tube_from_kept_dict(d: dict) -> Tube:
    """Rebuild a Tube from the per-tube dict persisted in predictions.json."""
    entries: list[TubeEntry] = []
    for e in d["entries"]:
        bbox = e["bbox"]
        if bbox is None:
            det: Detection | None = None
        else:
            det = Detection(
                class_id=0,
                cx=bbox[0],
                cy=bbox[1],
                w=bbox[2],
                h=bbox[3],
                confidence=e["confidence"] if e["confidence"] is not None else 0.0,
            )
        entries.append(
            TubeEntry(frame_idx=e["frame_idx"], detection=det, is_gap=e["is_gap"])
        )
    return Tube(
        tube_id=d["tube_id"],
        entries=entries,
        start_frame=d["start_frame"],
        end_frame=d["end_frame"],
    )


def _show_raw_frames_with_bboxes(image_paths: list[Path], tubes: list[Tube]) -> None:
    """Raw frames grid with tube bboxes overlaid (color-coded per tube)."""
    if tubes:
        annotated = draw_tubes_on_frames(image_paths, tubes)
    else:
        annotated = [cv2.imread(str(p)) for p in image_paths]
    n = len(annotated)
    rows = max(1, (n + FRAMES_PER_ROW - 1) // FRAMES_PER_ROW)
    fig, axes = plt.subplots(
        rows, FRAMES_PER_ROW, figsize=(FRAMES_PER_ROW * 2.4, rows * 2)
    )
    axes_list = axes.flatten() if rows * FRAMES_PER_ROW > 1 else [axes]
    for i, ax in enumerate(axes_list):
        ax.axis("off")
        if i < n and annotated[i] is not None:
            ax.imshow(cv2.cvtColor(annotated[i], cv2.COLOR_BGR2RGB))
            ax.set_title(f"f{i}", fontsize=7)
    plt.tight_layout()
    plt.show()


def show_error(record: dict) -> None:
    """Render the full diagnostic view for one FN/FP sequence."""
    label = record["label"]
    seq_id = record["sequence_id"]
    image_paths = _sequence_image_paths(label, seq_id)
    if not image_paths:
        print(f"[SKIP] no images for {seq_id}")
        return
    # Deployment may pad inputs so tube entries reference frame indices beyond
    # the on-disk frame count; mirror that padding for visualization so
    # plot_tube_summary's index lookups succeed.
    kept = record.get("kept_tubes", [])
    max_frame_idx = max(
        (e["frame_idx"] for t in kept for e in t["entries"]),
        default=-1,
    )
    if max_frame_idx >= len(image_paths):
        image_paths = _pad_paths_symmetrically(image_paths, max_frame_idx + 1)

    score = record["score"]
    score_str = f"{score:.2f}" if score is not None else "none"
    threshold = record["threshold"]
    winner_id = record["winner_tube_id"]
    kept_tube_dicts = record["kept_tubes"]

    print("=" * 100)
    print(
        f"{seq_id}  [{label}]   score={score_str}  threshold={threshold:.4f}  "
        f"tubes_kept={record['num_tubes_kept']}"
    )
    for d in kept_tube_dicts:
        fires = d["logit"] >= threshold
        mark = "FIRED" if fires else "     "
        win = "  <-- WINNER" if d["tube_id"] == winner_id else ""
        print(
            f"  T{d['tube_id']:>2}  frames {d['start_frame']:>2}..{d['end_frame']:<2}  "
            f"logit={d['logit']:+.3f}  {mark}{win}"
        )

    tubes = [_tube_from_kept_dict(d) for d in kept_tube_dicts]

    # View 1: raw frames with all tube bboxes.
    _show_raw_frames_with_bboxes(image_paths, tubes)

    # View 2: tube timeline + filmstrips. tube_labels colors the
    # filmstrip borders: True (fired) -> green, False -> red.
    if tubes:
        tube_labels = [d["logit"] >= threshold for d in kept_tube_dicts]
        plot_tube_summary(
            image_paths=image_paths,
            tubes=tubes,
            num_frames=len(image_paths),
            tube_labels=tube_labels,
            title=(
                f"{seq_id} — tube timeline "
                "(filmstrip border: green=fired, red=did not fire)"
            ),
        )
        plt.show()

## False Negatives (smoke sequences the model missed)

In [ ]:
for r in false_negatives[:MAX_ERRORS_TO_SHOW]:
    show_error(r)

## False Positives (fp sequences the model flagged)

In [ ]:
for r in false_positives[:MAX_ERRORS_TO_SHOW]:
    show_error(r)